# ベンチマーク回路の生成

このドキュメントでは quration で用意しているベンチマーク回路の生成方法について説明します。Qurationでは以下の回路をコンフィグファイルから生成し、後述するQurationの中間言語であるQuration-IRの形式で保存します。現在サポートされているベンチマーク回路の生成は以下の通りです。

- Qubitizationに基づくQuantum Phase Estimation
  - PREPAREサブルーチン回路
  - SELECTサブルーチン回路
- Period Finding
  - Modular Bimultiplyサブルーチン回路
- Trotter分解によるQuantum dynamics simulation
- その他のArithmetic回路
  - QROM / UncomputeQROM
　- Craig Adder / Cuccaro Adder

以下の例では、 `quration-algorithm` をコンパイルして得られたベンチマーク回路の生成プログラムを呼び出します。このため、`README.md`に従い、`./build/benchmark_generators/` フォルダに`create_qpe`などの実行ファイルが生成されていることを確認してください。以下では、`./build/benchmark_generators/`を一時的にパスに追加し呼び出します。

また、以降のコマンド例において JSON ファイルを指定するコマンドでは、`quration-algorithm/benchmark_generators/data` の下にあるデフォルトの JSON ファイルを使用するため、`quration-algorithm/benchmark_generators/data` へのパスを取得します。

In [ ]:
import os
import pathlib
import platform

project_root = pathlib.Path("../../../..").resolve()
algorithm_generator_path = project_root / "build" / "benchmark_generators"
sample_data_dir = project_root / "quration-algorithm" / "benchmark_generators" / "data"
os.environ["PATH"] = str(algorithm_generator_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

## Quantum Phase Estimation (QPE) / PREPARE / SELECT

Quantum Phase Estimation (量子位相推定)は、$H = \sum_i \alpha_i P_i$の形式で与えられたハミルトニアンに対して、所与の初期状態から定まる確率分布でハミルトニアンの固有値をサンプリングするアルゴリズムです。実装は[R. Babbush et al., "Encoding Electronic Spectra in Quantum Circuits with Linear T Complexity"](https://arxiv.org/abs/1805.03662) に基づく実装に対応しており、Linear Combination of Unitariesを用いてblock encodingされたハミルトニアンに対して、SELECTおよびPREPAREと呼ばれるサブルーチンを繰り返し適用することで上記を実装します。QPEの回路の生成には `build/benchmark_generators/create_qpe` という実行ファイルを実行します。

例として $H = 0.25 Z \otimes I + 0.25 Z \otimes I + 0.5 X \otimes X$に対する位相推定を行う場合は以下のように設定します。

```json
{
    "paulis": [
        ["Z", "I"],
        ["Z", "Z"],
        ["X", "X"]
    ],                                       // ハミルトニアンのパウリ項のリスト P_i
    "lcu_coefficients": [0.25, 0.25, 0.5],   // ハミルトニアンのパウリの係数 alpha_i
    "system_size": 2,                        // ハミルトニアンの量子ビット数
    "hadamard_size": 3,                      // 求める固有値の2進数での固定小数点精度
    "sub_bit_precision": 3                   // 係数 alpha_i を扱う際の2進数での固定小数点精度
}
```

次のコマンドを実行すると、QPE の回路の中間表現の JSON ファイルが出力されます。
ここでは、`quration_algorithm/benchmark_generators/data/sample_qpe.json` を指定する場合のコマンド例を示しています。

In [ ]:
!create_qpe --input {sample_data_dir / "sample_qpe.json"} --output {output_dir / "tutorial_1_IR_qpe.json"}

QPEを構成するPREPARE 回路 (上記論文のFig.11に対応する回路) のみを生成する際には `build/benchmark_generators/create_prepare` という実行ファイルを実行します。 PREPARE回路はハミルトニアン$H = \sum_i \alpha_i P_i$となる正係数 $\alpha_i$ に対して $U|0\rangle = \sum_i \alpha_i |i\rangle$ のように状態を生成します。PREPAREを生成するにはQPEのコンフィグのうち、"lcu_coefficients"と"sub_bit_precision"が必要になります。
次のようにオプションをつけてコマンドを実行すると、PREPARE 回路の中間表現の JSON ファイルが出力されます。  

In [ ]:
!create_prepare --input {sample_data_dir / "sample_qpe.json"} --output {output_dir / "tutorial_1_IR_prepare.json"}

QPEを構成するSELECT 回路 (上記論文のFig.5およびFig.7に対応する回路) の生成には `build/benchmark_generators/create_select` という実行ファイルを実行します。SELECT回路はハミルトニアン $H = \sum_i \alpha_i P_i$ となる実係数パウリ項$P_i \in \pm \{I,X,Y,Z\}^{\otimes n}$の列に対して、$U = \sum_i | i \rangle \langle i | \otimes P_i $となるユニタリ作用を実現する回路です。SELECT回路の生成には"pauli_strings"が必要になります。

In [ ]:
!create_select --input {sample_data_dir / "sample_qpe.json"} --output {output_dir / "tutorial_1_IR_select.json"}

## Period Finding / Modular Bimultiply

位数発見アルゴリズムはある整数$N$とこれに互いに素な値$x$について、$N$の法のもとで$x^r \equiv 1$ となるような整数$r$を求める量子アルゴリズムです。Shorの素因数分解では素因数分解したい対象となる整数を$N$として、$N$より小さな値$r$を選択し、$r$が互いに素でない場合に位数発見を行うことで素因数分解を行います。Qurationでは[C.Gidney, "Factoring with n+2 clean qubits and n-1 dirty qubits"](https://arxiv.org/abs/1706.07884)のFig.4に対応する回路を `build/benchmark_generators/create_period_finding` という実行ファイルで生成します。

例えば15の法のもとで$2^r \equiv 1$となる高々4 bitの値$r$を探すには、以下のように設定します。

```json
{
  "modulus": "15",         // 位数発見を行う際の法 N
  "coprime_integer": "2",  // 位数発見の際にNと互いに素となる x
  "depth": 4               // 位数発見の結果として得られる値 r の2進数での桁数
}
```

次のコマンドを実行すると、位数発見回路の中間表現の JSON ファイルが出力されます。ここでは、`quration_algorithm/benchmark_generators/data/sample_period_finding.json` を指定する場合のコマンド例を示しています。

In [ ]:
!create_period_finding --input {sample_data_dir / "sample_period_finding.json"} --output {output_dir / "tutorial_1_IR_period_finding.json"}

MultiControlledModBiMulImm 回路( https://arxiv.org/pdf/1706.07884 Sec.2.2およびhttps://arxiv.org/abs/1905.07682 を参照)の生成には `build/benchmark_generators/create_multi_controlled_mod_bi_mul_imm` という実行ファイルを実行します。この回路は整数$N$、積算する数$K$をもとに生成される回路であり、入力される二つの整数$(x,y)$から$(xK, yK^{-1})$を$N$の法の下で計算する回路を、いくつかの制御量子ビットの下で行う回路です。この時、積算する値$K$は即値として与えられます。この回路は https://arxiv.org/pdf/1706.07884 のFig.1.にある通り、位数発見に対応するModular Exponentiationに付随する最も主要なサブルーチンとなっています。

まず、生成に必要となる情報が書かれた以下のフォーマットの JSON ファイルを用意します。  

```json
{
  "modulus": "15",          // 法をとる値 N
  "multiplier": "2",        // 積算を行う値 K
  "num_control_qubits": 1,  // 制御量子ビットの数
  "num_system_qubits": 4    // 計算を行う際の整数の2進数での桁数
}
```
ここでは、`quration_algorithm/benchmark_generators/data/sample_period_finding.json` を指定する場合のコマンド例を示しています。

次のようにオプションで入力パラメータを設定してコマンドを実行すると、MultiControlledModBiMulImm 回路の中間表現のJSONファイルが出力されます。

In [ ]:
!create_multi_controlled_mod_bi_mul_imm --input {sample_data_dir / "sample_modular_bimultiply.json"} --output {output_dir / "tutorial_1_modular_bimultiply.json"}

## Quantum Dynamics Simulation (Trotter expansion)

量子ダイナミクスシミュレーションは、所与のハミルトニアン $H=\sum_i \alpha_i P_i$ 、初期状態 $|\psi_0 \rangle$ およびシミュレーションする時刻$T$から、$e^{iHt} |\psi_0 \rangle$という量子状態を生成する量子アルゴリズムです。Qurationではトロッター展開を用いてこれを近似的に計算するアルゴリズムを実装しています。トロッター展開ではトロッター数と呼ばれる十分大きな整数$M$について、$e^{iHt} = (e^{iHt/M})^M \simeq (\prod_i e^{i t \alpha_i P_i / M})^M$ とダイナミクスを近似することで、パウリの微小角回転を繰り返し上記のシミュレーションを行います。トロッター展開を用いた Hamiltonian の時間発展をシミュレーションする回路の生成には `build/benchmark_generators/create_trotter` という実行ファイルを実行します。 

入力で指定する Hamiltonian は Pauli 文字列から構成されることを仮定します。Hamiltonian は次のようなフォーマットの JSON ファイルを作成して量子ビット数と Hamiltonian を構成する Pauli 文字列と係数を指定します。例えば3量子ビットの縦磁場イジング模型は以下のように記述できます。

```json
{
    "num_qubits": 3,                // ハミルトニアンの量子ビット数
    "num_trotter_steps": 1,         // トロッター数
    "time": 1.0,                    // シミュレーションする時間
    "pauli_terms": [
        {
            "coeff": 1.0,
            "pauli_string": {
                "0": "X"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "0": "Z",
                "1": "Z"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "1": "X"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "1": "Z",
                "2": "Z"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "2": "X"
            }
        },
        {
            "coeff": 1.0,
            "pauli_string": {
                "2": "Z",
                "0": "Z"
            }
        }
    ]                              // パウリのリスト
}
```

次のようにコマンドを実行すると、指定した Hamiltonian の時間発展をシミュレートする回路の JSON ファイルが出力されます。  
ここでは、`quration_algorithm/examples/data/1d_ising_hamiltonian.json` を指定する場合のコマンド例を示しています。

In [ ]:
!create_trotter --input {sample_data_dir / "sample_trotter.json"} --output {output_dir / "tutorial_1_trotter.json"}

## Quantum Read-Only Memory (QROM)

Quantum Read-Only Memory (QROM)は、事前に指定された値を指定されたレジスタから読み出す操作であり、$x$に紐づいた整数$D_x$について、$U|x\rangle |0\rangle = |x\rangle |D_x\rangle$という操作を行うルーチンです。この回路はPREPARE回路やModular Bimultiply回路のサブルーチンとして利用されます。Uncompute QROMは上記の逆手順を行う処理です。QurationではQROMの実装例として $D_x = ax+b \mod 2^N$ として定義されるデータの読み出しを実現します。回路ではこの積算やmodを計算するのではなく、事前に計算された値$D_x$を読み出すので、$D_x$の値は何であっても効率には影響しないことに留意してください。QROM回路の生成には`build/benchmark_generators/create_qrom` という実行ファイルを実行します。 

例えば、入力となるアドレス$x$が5-bitで指定され、データ$D_x$が$D_x = 11x+3 \mod 2^6$となる設定でQROMを構築したい場合は以下のようにします。
```json
{
  "address_size": 5,           // アドレスxの桁数
  "value_size": 6,             // データ値の桁数
  "multiplier": "11",    // xに積算される値a
  "offset": "3"        // axに加算される値b
}
```
次のようにコマンドを実行すると、指定した QROM回路の JSON ファイルが出力されます。
ここでは、`quration_algorithm/benchmark_generators/data/sample_qrom.json` を指定する場合のコマンド例を示しています。


In [ ]:
!create_qrom --input {sample_data_dir / "sample_qrom.json"} --output {output_dir / "tutorial_1_qrom.json"}

同様に、QROMのuncompute回路を生成したい場合は以下のようにします。

In [ ]:
!create_uncompute_qrom --input {sample_data_dir / "sample_qrom.json"} --output {output_dir / "tutorial_1_uncompute_qrom.json"}

## Craig Adder / Cuccaro Adder

Craig AdderおよびCuccaro Adderは二つの入力値$(x,y)$について、$(x,x+y)$を出力する加算回路です。詳細は[C.Gidney, Halving the cost of quantum addition](https://arxiv.org/abs/1709.06648)を参照してください。FTQCにおいて実装する際は、Craig Adderの方が高速に実装できることが知られています。

5 bitの加算回路を生成したい場合は以下のようにJSONを作成します。

```json
{
  "size": 5
}
```

次のようにコマンドを実行すると、指定したビット幅のCraigによるAdd回路の JSON ファイルが出力されます。 ここでは、`quration_algorithm/benchmark_generators/data/sample_add.json` を指定する場合のコマンド例を示しています。

In [ ]:
!create_add_craig --input {sample_data_dir / "sample_add.json"} --output {output_dir / "tutorial_1_add_craig.json"}

次のようにコマンドを実行すると、指定したビット幅のCuccaroによるAdd回路の JSON ファイルが出力されます。 ここでは、`quration_algorithm/benchmark_generators/data/sample_add.json` を指定する場合のコマンド例を示しています。

In [ ]:
!create_add_cuccaro --input {sample_data_dir / "sample_add.json"} --output {output_dir / "tutorial_1_add_cuccaro.json"}